<a href="https://colab.research.google.com/github/Jeffrey1999/Deep-Learning/blob/main/physics_informed_CALCE%20PREDICTION2%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================
# CALCE CS2 DATA LOADER
# ============================

from google.colab import drive
import pandas as pd
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

DATA_DIR = "/content/drive/MyDrive/Datasets/BatteryLifePrediction"
cell_names = ['CS2_35', 'CS2_36', 'CS2_37', 'CS2_38']

# Verify directory exists
if not os.path.exists(DATA_DIR):
    print(f"ERROR: {DATA_DIR} not found. Check your Drive path.")
else:
    print(f"✓ Data directory found: {DATA_DIR}")
    print(f"Files in directory: {os.listdir(DATA_DIR)[:10]}")

# ============================
# Load and preprocess CALCE CS2
# ============================

def load_calce_cell(cell_name, data_dir):
    """
    Load a single CALCE CS2 CSV file and extract 17 features per cycle.

    Expected CSV structure (CALCE CS2):
    - Columns include: cycle, data_point, time, current, voltage,
                       discharge_capacity, charge_capacity, energy,
                       internal_resistance, impedance, etc.
    """
    filepath = os.path.join(data_dir, f"{cell_name}.csv")

    if not os.path.exists(filepath):
        print(f"WARNING: {filepath} not found")
        return None

    print(f"Loading {cell_name}...")
    df = pd.read_csv(filepath)
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()[:10]}...")  # first 10 cols

    return df


# Load all cells
cells_data = {}
for cell_name in cell_names:
    df = load_calce_cell(cell_name, DATA_DIR)
    if df is not None:
        cells_data[cell_name] = df

print(f"\n✓ Loaded {len(cells_data)} cells")

# ============================
# Feature extraction and cycle aggregation
# ============================

def extract_cycle_features(df, EOL_threshold=0.77, C0=1.1):
    """
    Extract cycle-level features from CALCE CS2 raw data.

    Assumptions:
    - df has columns: cycle, data_point, time, voltage, current,
                      discharge_capacity, charge_capacity, energy,
                      internal_resistance, impedance, etc.
    - One row per data point; one cycle may have many data points.
    - We aggregate within each cycle.

    Returns:
    - cycles_df: (num_cycles, num_features) with features including:
        - cycle_index, avg_voltage, min_voltage, max_voltage,
        - avg_current, avg_temperature (if available),
        - discharge_capacity, charge_capacity, energy,
        - internal_resistance, impedance, ...
    - RUL: remaining useful life (cycles until EOL_threshold)
    - EOL_idx: cycle index when capacity drops below threshold
    """

    # Map desired standard feature names to potential column names in the raw DataFrame
    feature_mapping = {
        'cycle': ['cycle'],
        'voltage': ['voltage'],
        'current': ['current'],
        'discharge_capacity': ['discharge_capacity', 'capacity'], # Map 'capacity' from raw data
        'charge_capacity': ['charge_capacity', 'capacity'],   # Map 'capacity' from raw data
        'energy': ['energy'],
        'internal_resistance': ['internal_resistance', 'resistance'], # Map 'resistance' from raw data
        'impedance': ['impedance'],
        'soh': ['SoH', 'soh'] # Add SoH as a potential feature
    }

    available_cols = {}
    for standard_feature, possible_cols in feature_mapping.items():
        for col in possible_cols:
            if col in df.columns:
                available_cols[standard_feature] = col
                break # Found a match, move to next standard feature

    print(f"  Available features: {list(available_cols.keys())}")

    # Ensure 'cycle' is present for grouping, otherwise fall back to index-based grouping
    if 'cycle' in available_cols:
        cycle_groups = df.groupby(available_cols['cycle'])
    else:
        # Fallback if 'cycle' column is not found (e.g., if using data_point or implicit cycles)
        # This might need adjustment based on specific dataset structure if 'cycle' is truly missing
        print("  WARNING: 'cycle' column not found, grouping by approximate cycle index.")
        cycle_groups = df.groupby(df.index // 100) # Assuming ~100 data points per cycle


    cycles_data = []
    for cycle_idx, group in cycle_groups:
        row = {'cycle_index': cycle_idx}

        # Voltage stats (if available)
        if 'voltage' in available_cols:
            V = group[available_cols['voltage']].values
            row['V_mean'] = float(np.nanmean(V)) if V.size > 0 else 0.0
            row['V_min'] = float(np.nanmin(V)) if V.size > 0 else 0.0
            row['V_max'] = float(np.nanmax(V)) if V.size > 0 else 0.0
            row['V_std'] = float(np.nanstd(V)) if V.size > 0 else 0.0

        # Current stats (if available)
        if 'current' in available_cols:
            I = group[available_cols['current']].values
            row['I_mean'] = float(np.nanmean(I)) if I.size > 0 else 0.0
            row['I_std'] = float(np.nanstd(I)) if I.size > 0 else 0.0

        # Capacity (if available)
        if 'discharge_capacity' in available_cols:
            row['discharge_capacity'] = float(group[available_cols['discharge_capacity']].iloc[-1])
        if 'charge_capacity' in available_cols:
            row['charge_capacity'] = float(group[available_cols['charge_capacity']].iloc[-1])

        # Energy (if available)
        if 'energy' in available_cols:
            row['energy'] = float(group[available_cols['energy']].iloc[-1])

        # Resistance (if available)
        if 'internal_resistance' in available_cols:
            row['internal_resistance'] = float(group[available_cols['internal_resistance']].iloc[-1])
        if 'impedance' in available_cols:
            row['impedance'] = float(group[available_cols['impedance']].iloc[-1])

        # SoH (State of Health, if available)
        if 'soh' in available_cols:
            row['SoH'] = float(group[available_cols['soh']].iloc[-1])

        cycles_data.append(row)

    cycles_df = pd.DataFrame(cycles_data)

    # Handle missing values after creating DataFrame (e.g., if a feature wasn't always present)
    cycles_df = cycles_df.fillna(0)

    # Compute RUL (assuming discharge_capacity is the key metric)
    # Prioritize 'discharge_capacity', then 'capacity' if only that was mapped
    capacity_col_for_rul = None
    if 'discharge_capacity' in cycles_df.columns:
        capacity_col_for_rul = 'discharge_capacity'
    elif 'capacity' in df.columns: # Check if original 'capacity' column was present
        capacity_col_for_rul = 'capacity' # Use original if mapped to both

    if capacity_col_for_rul and capacity_col_for_rul in cycles_df.columns:
        eol_idx = np.where(cycles_df[capacity_col_for_rul].values <= EOL_threshold)[0]
        if len(eol_idx) > 0:
            N_EOL = eol_idx[0]
        else:
            N_EOL = len(cycles_df)
    else:
        N_EOL = len(cycles_df) # Default to full length if no capacity column found

    cycles_df['RUL'] = N_EOL - cycles_df.index.values
    cycles_df['RUL'] = cycles_df['RUL'].clip(lower=0)  # RUL >= 0

    return cycles_df, N_EOL


# Extract features for all cells
all_cycles_data = {}
for cell_name, df in cells_data.items():
    print(f"\nExtracting features for {cell_name}...")
    cycles_df, N_EOL = extract_cycle_features(df, EOL_threshold=0.77)
    all_cycles_data[cell_name] = {
        'cycles_df': cycles_df,
        'N_EOL': N_EOL
    }
    print(f"  Cycles extracted: {len(cycles_df)}, EOL at cycle: {N_EOL}")
    print(f"  Feature columns: {cycles_df.columns.tolist()}")

# ============================
# Create windowed sequences
# ============================

def create_sequences(cycles_df, window_size=50, stride=1):
    """
    Create sliding windows of cycle features for CNN–Attention input.

    Args:
    - cycles_df: (num_cycles, num_features) DataFrame
    - window_size: T (number of cycles per window)
    - stride: step size for sliding window

    Returns:
    - X: (num_windows, T, num_features)
    - y: (num_windows,) with RUL target
    """
    # Extract numeric features (exclude cycle_index, RUL)
    feature_cols = [c for c in cycles_df.columns if c not in ['cycle_index', 'RUL']]
    X_data = cycles_df[feature_cols].values  # (num_cycles, num_features)
    y_data = cycles_df['RUL'].values          # (num_cycles,)

    X_windows = []
    y_windows = []

    for start_idx in range(0, len(X_data) - window_size + 1, stride):
        X_win = X_data[start_idx:start_idx + window_size]
        y_win = y_data[start_idx + window_size - 1]  # RUL at end of window
        X_windows.append(X_win)
        y_windows.append(y_win)

    X_windows = np.array(X_windows, dtype=np.float32)  # (num_windows, T, features)
    y_windows = np.array(y_windows, dtype=np.float32)

    return X_windows, y_windows, feature_cols


# Create sequences for all cells
WINDOW_SIZE = 50
all_sequences = {}

for cell_name, data in all_cycles_data.items():
    print(f"\nCreating sequences for {cell_name} (window_size={WINDOW_SIZE})...")
    X, y, feature_cols = create_sequences(data['cycles_df'], window_size=WINDOW_SIZE)
    all_sequences[cell_name] = {
        'X': X,
        'y': y,
        'feature_cols': feature_cols,
        'n_features': X.shape[2]
    }
    print(f"  Sequences created: {X.shape}")
    print(f"  Features: {len(feature_cols)} → {feature_cols}")

# ============================
# Combine and normalize
# ============================

# Concatenate all cells
X_all = np.concatenate([all_sequences[cell]['X'] for cell in cell_names], axis=0)
y_all = np.concatenate([all_sequences[cell]['y'] for cell in cell_names], axis=0)
n_features = X_all.shape[2]

print(f"\nCombined dataset shape: X={X_all.shape}, y={y_all.shape}")
print(f"Total features: {n_features}")

# Normalize features (z-score normalization across all samples and time steps)
X_mean = X_all.reshape(-1, n_features).mean(axis=0)
X_std = X_all.reshape(-1, n_features).std(axis=0) + 1e-8
X_normalized = (X_all - X_mean) / X_std

y_mean = y_all.mean()
y_std = y_all.std() + 1e-8
y_normalized = (y_all - y_mean) / y_std

print(f"Normalization done.")
print(f"  X: mean={X_mean}, std={X_std}")
print(f"  y: mean={y_mean:.2f}, std={y_std:.2f}")

# ============================
# Create PyTorch Dataset
# ============================

class CalceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create dataset and split
calce_dataset = CalceDataset(X_normalized, y_normalized)
train_size = int(0.7 * len(calce_dataset))
val_size = int(0.15 * len(calce_dataset))
test_size = len(calce_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    calce_dataset, [train_size, val_size, test_size]
)

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")
print(f"\n✓ Data pipeline ready! Input shape: ({WINDOW_SIZE}, {n_features})")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Data directory found: /content/drive/MyDrive/Datasets/BatteryLifePrediction
Files in directory: ['.git', '.gitattributes', 'CS2_35.csv', 'CS2_36.csv', 'CS2_37.csv', 'CS2_38.csv', 'Qore_learning.ipynb', 'lstm_learning.ipynb', 'readme.md', 'CS2_34']
Loading CS2_35...
  Shape: (846, 7)
  Columns: ['Unnamed: 0', 'cycle', 'capacity', 'SoH', 'resistance', 'CCCT', 'CVCT']...
Loading CS2_36...
  Shape: (936, 7)
  Columns: ['Unnamed: 0', 'cycle', 'capacity', 'SoH', 'resistance', 'CCCT', 'CVCT']...
Loading CS2_37...
  Shape: (972, 7)
  Columns: ['Unnamed: 0', 'cycle', 'capacity', 'SoH', 'resistance', 'CCCT', 'CVCT']...
Loading CS2_38...
  Shape: (958, 7)
  Columns: ['Unnamed: 0', 'cycle', 'capacity', 'SoH', 'resistance', 'CCCT', 'CVCT']...

✓ Loaded 4 cells

Extracting features for CS2_35...
  Available features: ['cycle', 'discharge_capacity', 'charge_capacity', 'in

# Task
Define a PyTorch LSTM model for Remaining Useful Life (RUL) prediction, taking an input sequence of shape (50, 4) and outputting a single RUL value.

## Define Model Architecture

### Subtask:
Define the PyTorch model architecture, considering the input shape (50, 4) from the preprocessed data. An LSTM model would be appropriate for this sequence prediction task.


**Reasoning**:
The subtask requires defining a PyTorch LSTM model. I will create a Python code block to define the `LSTM_RUL_Predictor` class, inheriting from `nn.Module`, and implement its `__init__` and `forward` methods as specified.



In [ ]:
import torch.nn as nn

class LSTM_RUL_Predictor(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=1):
        super(LSTM_RUL_Predictor, self).__init__()
        self.hidden_size = hidden_size

        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)

        # Linear layer to output RUL
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)

        # Initialize hidden state and cell state (optional, can be left to default if always starting fresh)
        # h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        # c0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)

        # Forward propagate LSTM
        # out: tensor of shape (batch_size, sequence_length, hidden_size)
        # _: contains (h_n, c_n), where h_n is the hidden state at the last time step
        out, _ = self.lstm(x) #, (h0, c0))

        # We are interested in the output of the last time step
        # out[:, -1, :] will give the output features from the last time step
        out = self.linear(out[:, -1, :])

        return out

print("LSTM_RUL_Predictor model class defined.")

LSTM_RUL_Predictor model class defined.


## Define Loss Function and Optimizer

### Subtask:
Set up the loss function (e.g., Mean Squared Error for regression) and the optimizer (e.g., Adam) for training the model.


**Reasoning**:
I need to import the necessary PyTorch modules, instantiate the `LSTM_RUL_Predictor` model with appropriate input and hidden sizes, and then define the Mean Squared Error loss function and the Adam optimizer as instructed. The `n_features` variable from the previous execution provides the `input_size`.



In [ ]:
import torch.optim as optim

# Instantiate the model
# n_features is available from previous steps as the input_size for the LSTM
input_size = n_features
hidden_size = 64 # A common choice for hidden layer size
output_size = 1 # We are predicting a single RUL value

model = LSTM_RUL_Predictor(input_size, hidden_size, output_size)

# Define Loss Function (Mean Squared Error for regression)
criterion = nn.MSELoss()

# Define Optimizer (Adam)
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model: {model}")
print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

Model: LSTM_RUL_Predictor(
  (lstm): LSTM(4, 64, batch_first=True)
  (linear): Linear(in_features=64, out_features=1, bias=True)
)
Loss function: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
